In [1]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [2]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out#, maxs, probs

In [3]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [4]:
# A subroutine that calculates $\tau_b$ coefficient for columns $i$ and $j$ of the input dataframe `df`.

def tau_b(df, i, j):
  import scipy.stats as stats
  dat = df.copy()
  keep_ind = []
  for k in range(len(df)):
    if df.isna().iloc[k,i]+df.isna().iloc[k,j] == 0:
      keep_ind.append(k)
  dat_out = dat.iloc[keep_ind,:]
  tau, p = stats.kendalltau(dat_out.iloc[:,i],dat_out.iloc[:,j])
  return tau

In [5]:
def tau_b_mat(df):
  import numpy as np
  n = df.shape[1]
  tau_mat = np.empty(shape=(n, n), dtype='double')
  for i in range(n):
    for j in range(n):
      tau_mat[i,j] = tau_b(df, i, j)
  return tau_mat

# Subroutine: Column-wise aggregation 
It takes a normalized dataframe `df` and the column label `i` and returns the aggregated column by weighted average:
$$
s_i = \frac{\sum_l w_l X_i}{\sum_l w_l}
$$`

In [6]:
def c_agg(df, w):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]

  agg_mat = np.empty(shape=(m, n), dtype='double')
  masked_df = np.ma.masked_array(df, np.isnan(df))

  for i in range(m):
    for j in range(n):
      agg_mat[i,j] = np.ma.average(masked_df[i,:], weights=w[:,j])
  return agg_mat

Subroutine: Take a column of a missing data and return the imputed one based on monotonization

In [21]:
# new optimization procedure (2023.04.18)
def monotonization(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  # print(x_out)
  # print(y_out)
  # print(K)

  # for j in range(n):
  #   if j <= (K[0]+K[1])/2:
  #     x_out[j] = y_out[0]
  #   elif j > (K[obs-2]+K[obs-1])/2:
  #     x_out[j] = y_out[obs-1]
  #   else:
  #     for i in range(1,obs-1):
  #       if (K[i-1]+K[i])/2 < j <= (K[i]+K[i+1])/2:
  #         x_out[j] = y_out[i]
  
  for j in range(n):
    if j <= K[0]:
      x_out[j] = y_out[0]
    for k in range(obs-1):
      if j >= K[k] and j < K[k+1]:
        x_out[j] = y_out[k+1]
      if j >= K[k+1]:
        x_out[j] = y_out[obs-1]
      
  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [8]:
def rmse_cal(df, corr_mat, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)
  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()
  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [9]:
def rmse_cal2(df, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)

  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()

  w = np.array(df.corr())
  w[w<0]=0
  for i in range(len(w)):
    w[i,i]=0

  corr_mat = w

  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [10]:
def nested_quantile(df, a_min, a_max, n_level, RMSE_max):
  import numpy as np
  Q = np.zeros(5)
  Q[0] = a_min
  Q[4] = a_max
  beta = abs(Q[0]-Q[4])/4
  for j in range(1,4):
    Q[j] = Q[0] + j*beta

  # RMSE_L = rmse_calculator(df,Q[0])
  # RMSE_R = rmse_calculator(df,Q[4])
  RMSE_L = rmse_cal2(df,Q[0])
  RMSE_R = rmse_cal2(df,Q[4])

  # When RMSE_M=1, it is temporary
  RMSE_M = 1

  for i in range(n_level):

    RMSE = pd.DataFrame([[Q[0],RMSE_L],[Q[1],None],[Q[2],RMSE_M],[Q[3],None],[Q[4],RMSE_R]])

    for k in range(1,4):
      if k != 2 or RMSE_M == 1:
        # RMSE.iloc[k,1] =  rmse_calculator(df,Q[k])
        RMSE.iloc[k,1] =  rmse_cal2(df,Q[k])
      else:
        RMSE.iloc[k,1] = RMSE_M
    
    RMSE_sorted = RMSE.sort_values(by=[1])
        
    #print('RMSE Sorted:')
    #display(RMSE_sorted)
    
    alpha = RMSE_sorted.iloc[0,0]

    if RMSE_sorted.iloc[0,0] == Q[0]:
      Q[4] = Q[1]
      RMSE_R = RMSE.iloc[1,1] 
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[4]:
      Q[0] = Q[3] 
      RMSE_L = RMSE.iloc[3,1]
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[1]:
      Q[4] = Q[2]
      RMSE_M = RMSE.iloc[1,1]
      RMSE_R = RMSE.iloc[2,1]
    elif RMSE_sorted.iloc[0,0] == Q[2]:
      Q[0] = Q[1]
      Q[4] = Q[3]
      RMSE_L = RMSE.iloc[1,1]
      RMSE_M = RMSE.iloc[2,1]
      RMSE_R = RMSE.iloc[3,1]
    elif RMSE_sorted.iloc[0,0] == Q[3]:
      Q[0] = Q[2]
      RMSE_L = RMSE.iloc[2,1]
      RMSE_M = RMSE.iloc[3,1]

    beta = abs(Q[0]-Q[4])/4
    for j in range(1,4):
      Q[j] = Q[0] + j*beta

  return alpha

In [11]:
def power_finder(df, corr_mat, step_size, init_power, max_power):
  import math
  current_power = init_power
  opt_power = current_power
  opt_rmse = rmse_cal(df, corr_mat, opt_power)
  n_iter = math.floor((max_power-init_power)/step_size)
  for i in range(n_iter):
    current_power += step_size
    new_rmse = rmse_cal(df, corr_mat, current_power)
    if opt_rmse > new_rmse:
      opt_power = current_power
      opt_rmse = new_rmse
  return opt_power

In [12]:
n_items_list = [ 100, 100, 100, 100]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 100, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]

for n_items, m_users in zip(n_items_list, m_users_list):
    # Your code using n_items and m_users
    # For example:
    print(f"n_items: {n_items}, m_users: {m_users}")
    # Run your analysis or processing using n_items and m_users here


n_items: 50, m_users: 50
n_items: 100, m_users: 100
n_items: 100, m_users: 300
n_items: 100, m_users: 5
n_items: 500, m_users: 100
n_items: 500, m_users: 500
n_items: 1000, m_users: 100
n_items: 1000, m_users: 200
n_items: 1000, m_users: 300
n_items: 1500, m_users: 1000
n_items: 1500, m_users: 2000


In [20]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 100, 100, 100, 100]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 100, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    powers = [] #for monimpute

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    corr_mat = df_orig.corr()
    temp_corr = np.copy(corr_mat)
    temp_corr[temp_corr < 0] = 0
    #print(np.round(temp_corr,2))
    #print(np.round(corr_mat,2))
    np.fill_diagonal(temp_corr, 0)
    orig_corr = np.copy(temp_corr)

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    a_min = 0
    a_max = 16
    n_level = 4
    RMSE_max = 1
    #col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        orig_corr = np.copy(temp_corr)
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)
        df_norm = normal3(masked_df)

        start = time_lib.time()

        power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
        powers.append(power)

        corr = np.power(orig_corr, power)  
        agg_mat = c_agg(df_norm, corr)

        df_agg_mat = pd.DataFrame(agg_mat)
        df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
        df_agg_mat.index = df.index
        df_agg = pd.concat([df, df_agg_mat], axis=1)
        #print(df_agg.shape)
        id_column = pd.DataFrame(range(mm))
        id_column.index = df_agg.index
        df_agg = pd.concat([id_column, df_agg], axis=1)
        #print(df_agg.shape)
        df_agg.columns = range(2*nn+1)
        
        

        for i in range(nn):
            df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
            index_i = df_agg_sorted.index
            vec = np.array(df_agg_sorted.iloc[:,i+1])
            out = monotonization(vec)
            out_list = []
            for i in range(len(out)):
                out_list.append(out[i])

            out_df = pd.DataFrame(out_list, index=index_i)
            df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

        df_agg.columns = range(3*nn+1)
        df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
        imputed = df_agg_sorted_final.iloc[:,2*nn+1:]

       
        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        url = './result/'

        if count<10:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
        else:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
        
        imputed.to_csv(filename)
        print("saved the result file as "+str(filename))
        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)
        

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        
result_summary_df.to_csv('result_summary_df_mono.csv')
result_detail_df.to_csv('result_detail_df_mono.csv')
        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-100_original_matrix.csv
./data/20230808_100-by-100_10_fold_test_data01.csv
./data/20230808_100-by-100_10_fold_test_data02.csv
./data/20230808_100-by-100_10_fold_test_data03.csv
./data/20230808_100-by-100_10_fold_test_data04.csv
./data/20230808_100-by-100_10_fold_test_data05.csv
./data/20230808_100-by-100_10_fold_test_data06.csv
./data/20230808_100-by-100_10_fold_test_data07.csv
./data/20230808_100-by-100_10_fold_test_data08.csv
./data/20230808_100-by-100_10_fold_test_data09.csv
./data/20230808_100-by-100_10_fold_test_data10.csv
sparsity: 0.2914
saved the result file as ./result/20230808_100-by-100_10_fold_test_data01_mono_imputed.csv
saved the result file as ./result/20230808_100-by-100_10_fold_test_data02_mono_imputed.csv
saved the result file as ./result/20230808_100-by-100_10_fold_test_data03_mono_imputed.csv
saved the result file as ./result/20230808_100-by-100

'n: 100 ,m: 100'

,0,1,2,3
0,0.598870,0.769291,0.459040,12.254653
1,0.567797,0.833616,0.511299,12.152371
2,0.515537,0.855359,0.559322,12.383173
3,0.526836,0.859477,0.552260,12.308501
4,0.547250,0.819371,0.519041,12.962185
5,0.545839,0.899773,0.561354,12.995097
6,0.545839,0.819371,0.521862,13.085621
7,0.521862,0.834719,0.547250,13.041009
8,0.544429,0.841451,0.533145,13.515741
9,0.583921,0.822806,0.496474,12.321169


saved 20230808_100-by-100the summary result file as./result/20230808_100-by-100_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_100-by-100_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.2914,0.549818,0.026759,0.835523,0.033673,0.526105,0.031954,12.701952,0.469232


...

,0,1,2,3
0,0.536998,0.933067,0.573784,50.395305
1,0.526004,0.914067,0.577590,50.414616
2,0.536152,0.946117,0.583087,50.510723
3,0.536575,0.923959,0.571247,51.081183
4,0.519662,0.975384,0.608879,50.579618
5,0.529387,0.931933,0.581818,51.018708
6,0.531078,0.946787,0.588584,50.869986
7,0.539112,0.929207,0.572516,50.520293
8,0.540575,0.936035,0.573542,51.631811
9,0.531699,0.943904,0.582418,50.711220


saved 20230808_100-by-500the summary result file as./result/20230808_100-by-500_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_100-by-500_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.29140,0.549818,0.026759,0.835523,0.033673,0.526105,0.031954,12.701952,0.469232
1,100.0,300.0,0.42020,0.573933,0.013751,0.767621,0.020731,0.476486,0.018513,33.469274,0.660815
2,100.0,500.0,0.52696,0.532724,0.006480,0.938046,0.016626,0.581346,0.011190,50.773346,0.386906


['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-700_original_matrix.csv
./data/20230808_100-by-700_10_fold_test_data01.csv
./data/20230808_100-by-700_10_fold_test_data02.csv
./data/20230808_100-by-700_10_fold_test_data03.csv
./data/20230808_100-by-700_10_fold_test_data04.csv
./data/20230808_100-by-700_10_fold_test_data05.csv
./data/20230808_100-by-700_10_fold_test_data06.csv
./data/20230808_100-by-700_10_fold_test_data07.csv
./data/20230808_100-by-700_10_fold_test_data08.csv
./data/20230808_100-by-700_10_fold_test_data09.csv
./data/20230808_100-by-700_10_fold_test_data10.csv
sparsity: 0.6093285714285714
saved the result file as ./result/20230808_100-by-700_10_fold_test_data01_mono_imputed.csv
saved the result file as ./result/20230808_100-by-700_10_fold_test_data02_mono_imputed.csv


UnboundLocalError: cannot access local variable 'k' where it is not associated with a value

In [24]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 500, 500, 500, 500, 500]#, , 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 100, 300, 500, 700 ]#, ,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    powers = [] #for monimpute

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    corr_mat = df_orig.corr()
    temp_corr = np.copy(corr_mat)
    temp_corr[temp_corr < 0] = 0
    #print(np.round(temp_corr,2))
    #print(np.round(corr_mat,2))
    np.fill_diagonal(temp_corr, 0)
    orig_corr = np.copy(temp_corr)

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    a_min = 0
    a_max = 16
    n_level = 4
    RMSE_max = 1
    #col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        orig_corr = np.copy(temp_corr)
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)
        df_norm = normal3(masked_df)

        start = time_lib.time()

        power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
        powers.append(power)

        corr = np.power(orig_corr, power)  
        agg_mat = c_agg(df_norm, corr)

        df_agg_mat = pd.DataFrame(agg_mat)
        df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
        df_agg_mat.index = df.index
        df_agg = pd.concat([df, df_agg_mat], axis=1)
        #print(df_agg.shape)
        id_column = pd.DataFrame(range(mm))
        id_column.index = df_agg.index
        df_agg = pd.concat([id_column, df_agg], axis=1)
        #print(df_agg.shape)
        df_agg.columns = range(2*nn+1)
        
        

        for i in range(nn):
            df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
            index_i = df_agg_sorted.index
            vec = np.array(df_agg_sorted.iloc[:,i+1])
            out = monotonization(vec)
            out_list = []
            for i in range(len(out)):
                out_list.append(out[i])

            out_df = pd.DataFrame(out_list, index=index_i)
            df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

        df_agg.columns = range(3*nn+1)
        df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
        imputed = df_agg_sorted_final.iloc[:,2*nn+1:]

       
        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        url = './result/'

        if count<10:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
        else:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
        
        imputed.to_csv(filename)
        print("saved the result file as "+str(filename))
        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)
        

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        
result_summary_df.to_csv('result_summary_df_mono.csv')
result_detail_df.to_csv('result_detail_df_mono.csv')
        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/500-by-50_original_matrix.csv
./data/20230808_500-by-50_10_fold_test_data01.csv
./data/20230808_500-by-50_10_fold_test_data02.csv
./data/20230808_500-by-50_10_fold_test_data03.csv
./data/20230808_500-by-50_10_fold_test_data04.csv
./data/20230808_500-by-50_10_fold_test_data05.csv
./data/20230808_500-by-50_10_fold_test_data06.csv
./data/20230808_500-by-50_10_fold_test_data07.csv
./data/20230808_500-by-50_10_fold_test_data08.csv
./data/20230808_500-by-50_10_fold_test_data09.csv
./data/20230808_500-by-50_10_fold_test_data10.csv
sparsity: 0.46664000000000005
saved the result file as ./result/20230808_500-by-50_10_fold_test_data01_mono_imputed.csv
saved the result file as ./result/20230808_500-by-50_10_fold_test_data02_mono_imputed.csv
saved the result file as ./result/20230808_500-by-50_10_fold_test_data03_mono_imputed.csv
saved the result file as ./result/20230808_500-by-50_1

'n: 500 ,m: 50'

,0,1,2,3
0,0.475619,0.944046,0.637659,23.361542
1,0.470368,0.927208,0.630158,22.785856
2,0.460615,0.973004,0.663166,22.637967
3,0.479370,0.910471,0.614404,22.925431
4,0.450113,0.957460,0.660165,22.779029
5,0.460615,0.904685,0.627907,23.097838
6,0.467766,0.950814,0.646177,22.716366
7,0.456522,0.948051,0.651424,22.718214
8,0.454273,0.957101,0.658171,22.583040
9,0.461019,0.954748,0.655172,22.549660


saved 20230808_500-by-50the summary result file as./result/20230808_500-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_500-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.46664,0.463628,0.009427,0.942759,0.021881,0.64444,0.016258,22.815494,0.25115


...

,0,1,2,3
0,0.487268,0.974485,0.636889,193.888222
1,0.495114,0.954217,0.619546,194.637192
2,0.500482,0.951762,0.614866,194.840589
3,0.493532,0.969107,0.627305,193.799410
4,0.500826,0.950828,0.613130,194.235907
5,0.481558,0.969107,0.639692,193.556362
6,0.500000,0.943855,0.610652,193.815416
7,0.483760,0.983696,0.643408,193.019090
8,0.493532,0.967259,0.627305,193.361929
9,0.500963,0.955953,0.616020,194.897170


saved 20230808_500-by-700the summary result file as./result/20230808_500-by-700_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_500-by-700_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.466640,0.463628,0.009427,0.942759,0.021881,0.644440,0.016258,22.815494,0.251150
1,500.0,100.0,0.530200,0.482546,0.010549,0.905237,0.019481,0.610345,0.016216,41.596394,0.438618
2,500.0,300.0,0.661573,0.507623,0.005755,0.864430,0.009571,0.571448,0.007650,101.895932,0.646383
3,500.0,500.0,0.740172,0.522699,0.006211,0.856982,0.012693,0.554475,0.009992,150.023129,0.589436
4,500.0,700.0,0.792409,0.493703,0.007294,0.962027,0.012522,0.624881,0.011862,194.005129,0.634721


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.466640,0.463628,0.009427,0.942759,0.021881,0.644440,0.016258,22.815494,0.251150
1,500.0,100.0,0.530200,0.482546,0.010549,0.905237,0.019481,0.610345,0.016216,41.596394,0.438618
2,500.0,300.0,0.661573,0.507623,0.005755,0.864430,0.009571,0.571448,0.007650,101.895932,0.646383
3,500.0,500.0,0.740172,0.522699,0.006211,0.856982,0.012693,0.554475,0.009992,150.023129,0.589436
4,500.0,700.0,0.792409,0.493703,0.007294,0.962027,0.012522,0.624881,0.011862,194.005129,0.634721


,n_items,m_users,acc,rmse,mad,time
0,500,50,0.475619,0.944046,0.637659,23.361542
1,500,50,0.470368,0.927208,0.630158,22.785856
2,500,50,0.460615,0.973004,0.663166,22.637967
3,500,50,0.479370,0.910471,0.614404,22.925431
4,500,50,0.450113,0.957460,0.660165,22.779029
5,500,50,0.460615,0.904685,0.627907,23.097838
6,500,50,0.467766,0.950814,0.646177,22.716366
7,500,50,0.456522,0.948051,0.651424,22.718214
8,500,50,0.454273,0.957101,0.658171,22.583040
9,500,50,0.461019,0.954748,0.655172,22.549660


In [25]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 1000, 1000, 1000, 1000, 1000]#, , 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 100, 300, 500, 700 ]#, ,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    powers = [] #for monimpute

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    corr_mat = df_orig.corr()
    temp_corr = np.copy(corr_mat)
    temp_corr[temp_corr < 0] = 0
    #print(np.round(temp_corr,2))
    #print(np.round(corr_mat,2))
    np.fill_diagonal(temp_corr, 0)
    orig_corr = np.copy(temp_corr)

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    a_min = 0
    a_max = 16
    n_level = 4
    RMSE_max = 1
    #col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        orig_corr = np.copy(temp_corr)
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)
        df_norm = normal3(masked_df)

        start = time_lib.time()

        power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
        powers.append(power)

        corr = np.power(orig_corr, power)  
        agg_mat = c_agg(df_norm, corr)

        df_agg_mat = pd.DataFrame(agg_mat)
        df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
        df_agg_mat.index = df.index
        df_agg = pd.concat([df, df_agg_mat], axis=1)
        #print(df_agg.shape)
        id_column = pd.DataFrame(range(mm))
        id_column.index = df_agg.index
        df_agg = pd.concat([id_column, df_agg], axis=1)
        #print(df_agg.shape)
        df_agg.columns = range(2*nn+1)
        
        

        for i in range(nn):
            df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
            index_i = df_agg_sorted.index
            vec = np.array(df_agg_sorted.iloc[:,i+1])
            out = monotonization(vec)
            out_list = []
            for i in range(len(out)):
                out_list.append(out[i])

            out_df = pd.DataFrame(out_list, index=index_i)
            df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

        df_agg.columns = range(3*nn+1)
        df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
        imputed = df_agg_sorted_final.iloc[:,2*nn+1:]

       
        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        url = './result/'

        if count<10:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
        else:
            filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
        
        imputed.to_csv(filename)
        print("saved the result file as "+str(filename))
        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)
        

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        
result_summary_df.to_csv('result_summary_df_mono.csv')
result_detail_df.to_csv('result_detail_df_mono.csv')
        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/1000-by-50_original_matrix.csv
./data/20230808_1000-by-50_10_fold_test_data01.csv
./data/20230808_1000-by-50_10_fold_test_data02.csv
./data/20230808_1000-by-50_10_fold_test_data03.csv
./data/20230808_1000-by-50_10_fold_test_data04.csv
./data/20230808_1000-by-50_10_fold_test_data05.csv
./data/20230808_1000-by-50_10_fold_test_data06.csv
./data/20230808_1000-by-50_10_fold_test_data07.csv
./data/20230808_1000-by-50_10_fold_test_data08.csv
./data/20230808_1000-by-50_10_fold_test_data09.csv
./data/20230808_1000-by-50_10_fold_test_data10.csv
sparsity: 0.63338
saved the result file as ./result/20230808_1000-by-50_10_fold_test_data01_mono_imputed.csv
saved the result file as ./result/20230808_1000-by-50_10_fold_test_data02_mono_imputed.csv
saved the result file as ./result/20230808_1000-by-50_10_fold_test_data03_mono_imputed.csv
saved the result file as ./result/20230808_1000-by-5

'n: 1000 ,m: 50'

,0,1,2,3
0,0.452264,0.979328,0.669940,48.404495
1,0.443535,1.016772,0.696672,46.518771
2,0.436989,1.025852,0.703219,44.704107
3,0.427169,1.041946,0.725586,47.490928
4,0.424986,1.039850,0.727769,46.426018
5,0.442444,1.053920,0.717949,44.552234
6,0.452810,0.984883,0.671031,43.795026
7,0.443535,1.018648,0.699400,44.801740
8,0.432079,1.002724,0.696672,43.342978
9,0.438931,1.014615,0.700109,43.293916


saved 20230808_1000-by-50the summary result file as./result/20230808_1000-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_1000-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.63338,0.439474,0.009462,1.017854,0.024133,0.700835,0.019767,45.333021,1.778334


...

,0,1,2,3
0,0.485070,0.965667,0.632146,281.184082
1,0.479520,0.961058,0.636808,282.612575
2,0.485292,0.964919,0.633145,280.500107
3,0.492285,0.949438,0.620602,281.620465
4,0.484848,0.974363,0.639472,282.263211
5,0.492951,0.961808,0.626263,283.649923
6,0.488289,0.966643,0.632923,282.111756
7,0.489345,0.963830,0.630189,281.535173
8,0.498224,0.952886,0.618979,283.738829
9,0.496448,0.951137,0.617869,282.089026


saved 20230808_1000-by-700the summary result file as./result/20230808_1000-by-700_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_1000-by-700_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.439474,0.009462,1.017854,0.024133,0.700835,0.019767,45.333021,1.778334
1,1000.0,100.0,0.686710,0.478343,0.008959,0.929588,0.012656,0.625203,0.011291,63.587506,0.913067
2,1000.0,300.0,0.785123,0.499077,0.006179,0.890118,0.011166,0.590354,0.008717,159.346893,4.027891
3,1000.0,500.0,0.837624,0.511652,0.004510,0.882141,0.010268,0.575652,0.006853,220.217933,9.788859
4,1000.0,700.0,0.871296,0.489228,0.005806,0.961175,0.007837,0.628839,0.007576,282.130515,1.017398


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.439474,0.009462,1.017854,0.024133,0.700835,0.019767,45.333021,1.778334
1,1000.0,100.0,0.686710,0.478343,0.008959,0.929588,0.012656,0.625203,0.011291,63.587506,0.913067
2,1000.0,300.0,0.785123,0.499077,0.006179,0.890118,0.011166,0.590354,0.008717,159.346893,4.027891
3,1000.0,500.0,0.837624,0.511652,0.004510,0.882141,0.010268,0.575652,0.006853,220.217933,9.788859
4,1000.0,700.0,0.871296,0.489228,0.005806,0.961175,0.007837,0.628839,0.007576,282.130515,1.017398


,n_items,m_users,acc,rmse,mad,time
0,1000,50,0.452264,0.979328,0.669940,48.404495
1,1000,50,0.443535,1.016772,0.696672,46.518771
2,1000,50,0.436989,1.025852,0.703219,44.704107
3,1000,50,0.427169,1.041946,0.725586,47.490928
4,1000,50,0.424986,1.039850,0.727769,46.426018
5,1000,50,0.442444,1.053920,0.717949,44.552234
6,1000,50,0.452810,0.984883,0.671031,43.795026
7,1000,50,0.443535,1.018648,0.699400,44.801740
8,1000,50,0.432079,1.002724,0.696672,43.342978
9,1000,50,0.438931,1.014615,0.700109,43.293916
